# Make a dataset from a jsonl file

In [5]:
import pandas as pd    
jsonObj = pd.read_json(path_or_buf="../records_export_full/stift-heiligenkreuz__2026-08-05T06-46-32Z.jsonl", lines=True)

# get only the necessary columns
df = pd.DataFrame(jsonObj)
del df["kp_id"]
del df["src_id"]
del df["institution_id"]
df


,content,pipeline_id
0,{'id': 'https://id.kulturpool.at/3298d5ec-2b53...,stift-heiligenkreuz
1,{'id': 'https://id.kulturpool.at/597b9d95-9e1d...,stift-heiligenkreuz
2,{'id': 'https://id.kulturpool.at/e4c4a5e7-8a38...,stift-heiligenkreuz
3,{'id': 'https://id.kulturpool.at/13cda2e6-2609...,stift-heiligenkreuz
4,{'id': 'https://id.kulturpool.at/500a2ffa-e9d3...,stift-heiligenkreuz
...,...,...
2390,{'id': 'https://id.kulturpool.at/7ae2102b-1e97...,stift-heiligenkreuz
2391,{'id': 'https://id.kulturpool.at/17c5cd2a-4f60...,stift-heiligenkreuz
2392,{'id': 'https://id.kulturpool.at/ed46a3d5-93c2...,stift-heiligenkreuz
2393,{'id': 'https://id.kulturpool.at/ad4c7e71-d403...,stift-heiligenkreuz


In [6]:
mydict = df["content"].to_list()
mydict[:5]

[{'id': 'https://id.kulturpool.at/3298d5ec-2b53-4c3a-8fbb-044356f17ecf/aggregation',
  'type': 'Aggregation',
  'hasView': [{'id': 'https://assets.museum-digital.org/at-noe/images/3/259-0215/entwurf_der_beschriftung_/entwurf-der-beschriftung-des-buchumschlages-fuer-arthur-roesslers-teschner-busch-von-1947-2.jpg',
    'type': 'WebResource',
    'creator': ['Erwin Pokorny'],
    'dcRights': ['Stift Heiligenkreuz'],
    'edmRights': 'http://creativecommons.org/licenses/by-nc-sa/4.0/'}],
  '@context': 'https://api.kulturpool.at/ns/v1/edm.json',
  'provider': 'Kulturpool',
  'edmRights': 'http://creativecommons.org/licenses/by-nc-sa/4.0/',
  'isShownAt': 'https://noe.at.museum-digital.org/object/259',
  'isShownBy': {'id': 'https://assets.museum-digital.org/at-noe/images/3/259-0215/entwurf_der_beschriftung_/entwurf-der-beschriftung-des-buchumschlages-fuer-arthur-roesslers-teschner-busch-von-1947.jpg',
   'type': 'WebResource',
   'creator': ['Erwin Pokorny'],
   'dcRights': ['Stift Heiligen

In [9]:
from collections import Counter

def find_key(data, key):
    """Yield every value stored under `key`, at any depth."""
    if isinstance(data, dict):
        for k, v in data.items():
            if k == key:
                yield v
            else:
                yield from find_key(v, key)
    elif isinstance(data, list):
        for item in data:
            yield from find_key(item, key)

def unpack(v):
    if isinstance(v, list):
        for item in v:
            yield from unpack(item)
    elif isinstance(v, dict):
        if "@value" in v:
            yield v["@value"], v.get("@language")
    elif v is not None:
        yield v, None

value_counts = Counter()
value_langs = {}

for d in mydict:
    for medium in find_key(d, "dcType"):
        for value, lang in unpack(medium):
            value_counts[value] += 1
            value_langs[value] = lang

print(value_counts)
print(value_langs)

Counter({'Zeichnung': 746, 'Aquarell': 159, 'Lithografie': 148, 'Holzschnitt': 139, 'Radierung': 127, 'Kupferstich': 120, 'Gemälde': 93, 'Bozzetto': 77, 'Exlibris': 49, 'Aquatinta': 45, 'Buchmalerei': 40, 'Krippenfigur': 39, 'Druckgrafik': 30, 'Gouache': 28, 'Holzstich': 27, 'Porträtzeichnung': 26, 'Lichtdruck': 20, 'Chromolithografie': 17, 'Glasscheibe': 16, 'Kaltnadelradierung': 15, 'Stahlstich': 15, 'Linolschnitt': 14, 'Malerei': 14, 'Anhänger (Schmuck)': 14, 'Bleistiftzeichnung': 13, 'Ölbild': 13, 'Glaskunst, Glasfenster': 13, 'Kolorierter Kupferstich': 12, 'Skulptur': 11, 'Krippe': 11, 'Bronzeskulptur': 10, 'Musikinstrument': 10, 'Gobelinmalerei': 10, 'Farbdruck': 9, 'Künstlerpostkarte': 7, 'Radio': 7, 'Glasfenster': 7, 'Holzskulptur': 7, 'Aktstudie': 6, 'Fresko': 6, 'Altarbild': 6, 'Reliquiar': 6, 'Glaskunst': 5, 'Neujahrskarte': 5, 'Entwurfszeichnung': 4, 'Heliogravüre': 4, 'Medaillon': 4, 'Ölgemälde': 4, 'Statuette': 4, 'Edition (Druckwerk)': 3, 'Schrifttafel': 3, 'Kolorierte R

In [7]:
from collections import Counter

def find_values(data, key=None):
    if isinstance(data, dict):
        if "@value" in data and key is None:
            yield data
        for k, v in data.items():
            if key is not None and k == key:
                yield from find_values(v)
            else:
                yield from find_values(v, key)
    elif isinstance(data, list):
        for item in data:
            yield from find_values(item, key)


value_counts = Counter()
value_langs = {}

for d in mydict:
    for obj in find_values(d, key="dcType"):
        value = obj["@value"]
        lang = obj.get("@language")   # not every @value has a language
        value_counts[value] += 1
        value_langs[value] = lang

print(dict(value_counts))
print(value_langs)

{}
{}


In [29]:
data = {"word": value_counts.keys(),
        "count": value_counts.values(),
        #"lang": value_langs.values(),
        }
df = pd.DataFrame(data)
df = df.sort_values("count", ascending=False)
df

,word,count
1,Papier / Holzschnitt,129
33,Papier / Lithografie,112
40,Papier / Radierung,102
21,Papier / Kupferstich,101
197,Öl auf Leinwand,97
...,...,...
686,"Aquarell, schwarze Tusche, schwarze Kreide auf...",1
685,"Kreiden, Aquarellfarben, schwarze Tusche, Deck...",1
684,"Aquarell, Deckweiß, schwarze Tusche, schwarze ...",1
713,"Aquarell, schwarze Tusche, blaue (Öl)kreide, s...",1


In [5]:
df.loc[df['lang'] == "de", 'word']

11086    Papierschneidemaschine
22366              Abgaskrümmer
11280               Guttapercha
11029             Arbeitskasten
10854           Seitenschneider
                  ...          
4564              Prothesenteil
6804          Bestrahlungslampe
2411                Gewichtsatz
4860                 Übertrager
4563             Blattschreiber
Name: word, Length: 5850, dtype: str

In [ ]:
en_list = df.loc[df['lang'] == "de", 'word'].to_list()

In [30]:
stift_list = df["word"].to_list()
print(len(stift_list))

721


In [32]:
from sentence_transformers import SentenceTransformer
from sklearn.cluster import AgglomerativeClustering

model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(stift_list)

clustering = AgglomerativeClustering(
    distance_threshold=0.8,  
    n_clusters=None,
    linkage='ward'
)

clustering.fit(embeddings)

# Step 4: Group words by cluster labels
labels = clustering.labels_
clusters = {}
for word, label in zip(stift_list, labels):
    clusters.setdefault(label, []).append(word)

# Step 5: Print clusters
for label, cluster_words in clusters.items():
    print(f"Cluster {label}: {cluster_words}")
print(len(clusters))

Cluster 36: ['Papier / Holzschnitt', 'Papier / Inkunabel und Holzschnitt', 'Pappe / Holzschnitt']
Cluster 105: ['Papier / Lithografie', 'Papier / Tonlithografie', 'Papier / Farblithografie']
Cluster 21: ['Papier / Radierung', 'Papier / Ätzradierung', 'Papier / Farbradierung', 'Papier / Steinradierung', 'Papier / Radierung (Ätzverfahren)', 'Papier / Umrissradierung']
Cluster 58: ['Papier / Kupferstich', 'Papier, Karton / Kupferstich', 'Papier/ Kupferstich', 'Papier / Buchdruck und Kupferstich', 'Papier / Buchdruck, Kupferstich']
Cluster 55: ['Öl auf Leinwand', 'Öl auf Leinwand; Ochsenaugenrahmen']
Cluster 77: ['Terrakotta', 'Terrakotta - Schlagmetall']
Cluster 0: ['Papier / Klischeedruck', 'Papier', 'Papier / Schabkunst', 'Papier / Kreide', 'Papier / Stich']
Cluster 9: ['Papier / Bleistift', 'Papier (Heft gebunden) / Bleistift', 'Bleistift / Papier']
Cluster 11: ['Aquarell, auf Papier.', 'Aquarell, Gouache, auf Papier.', 'Aquarell auf Papier.', 'Aquarell auf Papier', 'Aquarell auf Papie

# Datenset GND
https://sta.dnb.de/doc/RDA-E-W075

object_classe = [""]

In [35]:
from transformers import pipeline
clf = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

clf("Paper / Print", candidate_labels=["book", "photo", "print"])

Device set to use cpu


{'sequence': 'Paper / Print',
 'labels': ['print', 'book', 'photo'],
 'scores': [0.7980901002883911, 0.12584537267684937, 0.0760645642876625]}

In [3]:
import requests
import xml.etree.ElementTree as ET

URL = "https://www.semantics.gr/authorities/vocabularies/ekt-item-types/xml"

NS = {"skos": "http://www.w3.org/2004/02/skos/core#"}
XML_LANG = "{http://www.w3.org/XML/1998/namespace}lang"


def get_en_pref_labels(url: str) -> list[str]:
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    root = ET.fromstring(response.content)

    return [
        label.text.strip()
        for label in root.findall("skos:Concept/skos:prefLabel", NS)
        if label.get(XML_LANG) == "en" and label.text
    ]

labels = get_en_pref_labels(URL)
print(labels)
print(f"\nTotal: {len(labels)}")

['Card', 'Telecommunication equipment', 'Manual', 'Proverb', 'Architectural drawing', 'Chart', 'Icon', 'Doll', 'Dissertation or thesis (original)', 'Jewellery', 'Announcement', 'Biography', 'Index', 'Folktale', 'Play', 'Note', 'Bone', 'Document', 'Model', 'Souvenir', 'Newsreels', 'Identifying marking', 'Machine', 'Murals', 'Short story', 'Frame', 'Images and ornament for layout features', 'Concert videorecording', 'Performance recording', 'Mask', 'Religious visual works', 'Sculpture', 'Costume (mode of fashion)', 'Television program', 'Statistics', 'Music track', 'Book part', 'Radio Program', 'Concert', 'Photonegative', 'Leaflet', 'Personal accessory', 'Coin', 'Arm', 'Stamp', 'Sketch', 'Film', 'Interview', 'Ostracon', 'Printed material', 'Newspaper part', 'Autobiography', 'Print', 'Catalogue', 'Needlework', 'Seal', 'Intangible cultural heritage', 'Tools and equipment', 'Passport', 'Identity card', '3D model from scanning', 'Theater play videorecording', 'Event', 'Mixed media work of ar

In [12]:
import torch
from transformers import CLIPModel, CLIPTokenizer

MODEL_NAME = "openai/clip-vit-base-patch32"
TEMPLATE = "a photo of a {}"

device = "cuda" if torch.cuda.is_available() else "cpu"
model = CLIPModel.from_pretrained(MODEL_NAME).to(device).eval()
tokenizer = CLIPTokenizer.from_pretrained(MODEL_NAME)


@torch.no_grad()
def embed(words):
    texts = [TEMPLATE.format(w) for w in words]
    inputs = tokenizer(texts, padding=True, return_tensors="pt").to(device)
    feats = model.get_text_features(**inputs)
    return feats / feats.norm(dim=-1, keepdim=True)  # unit vectors -> dot product = cosine sim


reference_words = labels
ref_emb = embed(reference_words)

def check(new_words, top_k=3):
    sims = embed(new_words) @ ref_emb.T
    for word, row in zip(new_words, sims):
        scores, idx = row.topk(min(top_k, len(reference_words)))
        matches = ", ".join(f"{reference_words[i]} ({s:.3f})" for s, i in zip(scores.tolist(), idx.tolist()))
        print(f"{word:>12} -> {matches}")

check(["Paper / Print"])
while True:
    w = input("\nWord to check (blank to quit): ").strip()
    if not w:
        break
    check([w])

Paper / Print -> Print (0.965), Printed material (0.959), Printing proof (0.938)
tape recorder -> Videorecording (0.838), Telephone (0.830), Radio Program (0.829)
